# IAM-SRA2: temporary Colab GPU backend

**Prepared for the current local checkout; not executed or GPU-validated in Colab.**

The Streamlit application, privacy screening, session records and original project documents stay on davinci. Only Qwen inference runs here. The application sends screened prompts and selected source-derived passages to this Google-hosted runtime, so this changes the processing boundary. Use synthetic meeting examples and only material permitted on this service. Screening does not guarantee anonymity.

No repository, private PDFs, corpus, session exports or existing SSH private keys need to be uploaded. This notebook creates its own temporary SSH key. Never share the private key or add it to notebook output.

Use a **paid Colab runtime with L4 (24 GB) or A100**, subject to availability. The current FP16 Qwen3-8B needs more memory than a normal 16 GB T4 provides. Do not use a CPU/TPU or substitute a quantised model in this procedure.

**First check connectivity:** davinci must have an SSH hostname/port reachable from Colab, and its SSH policy must permit a loopback reverse forward. An internal hostname such as `davinci` may not resolve outside the university network. If access requires a VPN or unavailable jump host, stop here; this notebook cannot create that network access. Do not open public model ports or disable host-key verification.

Google documents restrictions on SSH/web-interface use in the free tier; paid runtimes with a positive compute balance remove those particular restrictions, but hardware availability and session lifetimes remain variable. This is a temporary experiment, not reliable meeting hosting.

References: [Colab FAQ](https://research.google.com/colaboratory/faq.html), [vLLM 0.8.5 GPU installation](https://docs.vllm.ai/en/v0.8.5/getting_started/installation/gpu.html), [uv Python environments](https://docs.astral.sh/uv/pip/environments/), [OpenSSH forwarding](https://man.openbsd.org/ssh#R).

## 1. Select and check the GPU

In Colab choose **Runtime → Change runtime type → L4 or A100 GPU**. Run this cell before installing packages. It refuses busy or unsupported hardware; it does not stop any GPU process.

In [ ]:
import base64
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import re
import shutil
import socket
import subprocess
import sys
import time
import urllib.request

gpu_tool = shutil.which('nvidia-smi')
if not gpu_tool:
    raise SystemExit('No NVIDIA GPU tool is available in this runtime. In hosted Google Colab choose Runtime > Change runtime type > L4 or A100 GPU, save, reconnect, then run this cell again. A CPU, TPU or local runtime will not provide the required Colab GPU. Do not install NVIDIA drivers or a Python nvidia-smi package to fix this.')
try:
    inventory = subprocess.check_output([
        gpu_tool, '--query-gpu=name,memory.total,memory.used',
        '--format=csv,noheader,nounits'
    ], text=True, stderr=subprocess.DEVNULL, timeout=20).strip().splitlines()
except (OSError, subprocess.SubprocessError):
    raise SystemExit('The NVIDIA GPU query could not complete. Reconnect to a hosted L4 or A100 GPU runtime and retry this cell. No GPU workload was changed.') from None
if len(inventory) != 1:
    raise RuntimeError('Use a single-GPU L4 or A100 runtime.')
gpu_name, gpu_total, gpu_used = [v.strip() for v in inventory[0].split(',')]
if not ('A100' in gpu_name or re.search(r'\bL4\b', gpu_name)):
    raise RuntimeError('Select an L4 or A100 runtime; no GPU workload was changed.')
if int(gpu_total) < 22000 or int(gpu_used) > 2048:
    raise RuntimeError('This GPU is too small or already occupied. Do not stop other workloads.')
if shutil.disk_usage('/content').free < 40 * 1024**3:
    raise RuntimeError('Allow at least 40 GiB of runtime disk space for packages and model files.')
print(json.dumps({'gpu': gpu_name, 'memory_mib': int(gpu_total),
                  'used_mib': int(gpu_used), 'notebook_python': sys.version.split()[0]}))


## 2. Check SSH reachability and prepare a temporary key

Enter davinci's **externally reachable SSH hostname**, not a model API URL. Ask the server administrator if you do not know it. This step makes only a connection check and creates a key inside this Colab runtime.

In [ ]:
ssh_host = input('Davinci SSH hostname (reachable from Colab): ').strip()
ssh_port = int(input('SSH port [22]: ').strip() or '22')
ssh_user = input('SSH username [evouvali]: ').strip() or 'evouvali'
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9.-]*', ssh_host):
    raise ValueError('Enter a DNS hostname or IPv4 address, without a URL or shell syntax.')
if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_-]*', ssh_user) or not 1 <= ssh_port <= 65535:
    raise ValueError('Invalid SSH username or port.')
try:
    with socket.create_connection((ssh_host, ssh_port), timeout=10):
        pass
except socket.gaierror:
    raise SystemExit('Colab cannot resolve this SSH hostname. Use the actual HostName/IP from the SSH configuration on the computer you normally connect from, not an alias such as davinci or your username. For a university-internal/VPN-only address, ask the administrator for an approved externally reachable SSH endpoint. Changing the port will not fix DNS resolution.') from None
except ConnectionRefusedError:
    raise SystemExit('The SSH connection was refused. Check the actual SSH hostname and port with the server administrator. No key or tunnel was created.') from None
except TimeoutError:
    raise SystemExit('The SSH connection timed out. The address may require a VPN or approved gateway that this Colab runtime cannot reach. Confirm the access route before downloading the model. No key or tunnel was created.') from None
except OSError:
    raise SystemExit('This Colab runtime could not connect to the SSH endpoint. Check the hostname, port and approved network access with the administrator. No key or tunnel was created.') from None
key_path = Path('/content/iam_colab_tunnel_key')
if not key_path.exists():
    subprocess.check_call(['ssh-keygen', '-q', '-t', 'ed25519', '-N', '', '-f', str(key_path)])
key_path.chmod(0o600)
public_key = Path(str(key_path) + '.pub').read_text().strip()
restricted_key = 'command="/bin/false",restrict,port-forwarding,permitlisten="127.0.0.1:18000" ' + public_key
print('On davinci, add the following PUBLIC key line to ~/.ssh/authorized_keys:')
print(restricted_key)
print('The private key stays inside this Colab runtime. Remove this exact authorized-key line when finished.')


On **davinci**, run:

```bash
mkdir -p ~/.ssh
chmod 700 ~/.ssh
touch ~/.ssh/authorized_keys
chmod 600 ~/.ssh/authorized_keys
nano ~/.ssh/authorized_keys
```

Append the public-key line printed above, preserving existing keys, and save. The key is limited to forwarding with a denied shell command. The forwarded model endpoint will listen on davinci's loopback port **18000**, not the public network. Loopback is still accessible to other users on that shared host; use synthetic examples.

Also obtain the SSH host fingerprint **in your already trusted davinci terminal**:

```bash
ssh-keygen -lf /etc/ssh/ssh_host_ed25519_key.pub
```

Copy just the `SHA256:...` fingerprint into the next cell. Do not accept an unverified fingerprint obtained from the network scan itself.

In [ ]:
expected_fingerprint = input('Trusted davinci ED25519 host fingerprint (SHA256:...): ').strip()
if not re.fullmatch(r'SHA256:[A-Za-z0-9+/]{43}', expected_fingerprint):
    raise ValueError('Enter the SHA256 fingerprint from your trusted davinci terminal.')
scan = subprocess.run(['ssh-keyscan', '-T', '10', '-p', str(ssh_port), '-t', 'ed25519', ssh_host],
                      capture_output=True, text=True, timeout=30, check=False)
verified = []
for row in scan.stdout.splitlines():
    parts = row.split()
    if len(parts) != 3 or parts[1] != 'ssh-ed25519':
        continue
    fingerprint = 'SHA256:' + base64.b64encode(hashlib.sha256(base64.b64decode(parts[2])).digest()).decode().rstrip('=')
    if fingerprint == expected_fingerprint:
        verified.append(row)
if not verified:
    raise RuntimeError('Host key could not be verified. Stop and check with the server administrator.')
known_hosts = Path('/content/iam_colab_known_hosts')
known_hosts.write_text('\n'.join(verified) + '\n')
known_hosts.chmod(0o600)
if 'tunnel' in globals() and tunnel.poll() is None:
    raise RuntimeError('This notebook already has an active tunnel; use it or run the cleanup cell first.')
tunnel_log = Path('/content/iam_colab_ssh.log')
with tunnel_log.open('w') as log:
    tunnel = subprocess.Popen([
        'ssh', '-N', '-T', '-i', str(key_path), '-p', str(ssh_port),
        '-o', 'BatchMode=yes', '-o', 'IdentitiesOnly=yes',
        '-o', 'StrictHostKeyChecking=yes', '-o', 'UserKnownHostsFile=' + str(known_hosts),
        '-o', 'ExitOnForwardFailure=yes', '-o', 'ConnectTimeout=15',
        '-o', 'ServerAliveInterval=30', '-o', 'ServerAliveCountMax=3',
        '-R', '127.0.0.1:18000:127.0.0.1:8000', ssh_user + '@' + ssh_host
    ], stdin=subprocess.DEVNULL, stdout=log, stderr=log)
time.sleep(3)
if tunnel.poll() is not None:
    raise RuntimeError('The SSH forward did not start. Check the temporary SSH log and server forwarding policy.')
print('SSH process started. Confirm loopback-only binding on davinci before downloading the model.')


On **davinci**, run `ss -ltn 'sport = :18000'`. Continue only if the listener is `127.0.0.1:18000`. If there is no listener, the tunnel is not ready. If it shows `0.0.0.0`, `[::]` or a public address, run the cleanup cell and ask the administrator to correct SSH `GatewayPorts`; do not expose the unauthenticated model API publicly. A site can enforce a wildcard bind despite the requested loopback address.

If the tunnel fails, fix the hostname, approved access, key or forwarding policy **before** installing the model. No notebook setting can overcome a VPN-only network or a server prohibition on forwarding.

In [ ]:
if input('Verified that davinci listens ONLY on 127.0.0.1:18000? Type yes: ').strip().lower() != 'yes':
    raise RuntimeError('Stop here and verify the SSH listener on davinci.')


## 3. Install an isolated copy of the existing model runtime

Colab's default Python/PyTorch versions can differ from this project. Use a separate Python **3.10** environment and the project's existing pins. This changes only the Colab runtime; it does not install or change anything on davinci. Downloading packages and the approximately 16 GB model takes time.

In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'uv'])
uv = shutil.which('uv')
if not uv:
    raise RuntimeError('uv is not on PATH; restart this Colab runtime and retry setup.')
subprocess.check_call([uv, 'python', 'install', '3.10'])
venv = Path('/content/iam_model_env')
model_python = venv / 'bin/python'
if not model_python.exists():
    subprocess.check_call([uv, 'venv', '--python', '3.10', '--seed', str(venv)])
pins = ['vllm==0.8.5', 'torch==2.6.0', 'torchaudio==2.6.0', 'torchvision==0.21.0',
        'xformers==0.0.29.post2', 'transformers==4.51.3', 'pydantic==2.11.4']
subprocess.check_call([uv, 'pip', 'install', '--python', str(model_python), *pins])
subprocess.check_call([str(model_python), '-m', 'pip', 'check'])
version_code = "import importlib.metadata,json,sys; print(json.dumps({'python':sys.version.split()[0], 'packages':{p:importlib.metadata.version(p) for p in ['vllm','torch','xformers','transformers','pydantic']}}))"
subprocess.check_call([str(model_python), '-c', version_code])
Path('/content/iam_colab_packages.txt').write_text(subprocess.check_output([str(model_python), '-m', 'pip', 'freeze'], text=True))


## 4. Download the exact model revision and start localhost inference

The command retains the project's FP16, V0/XFormers, XGrammar, 4096-token context and single-sequence configuration. It runs vLLM directly because the existing davinci launcher intentionally accepts only A30/V100 hardware. The notebook does not weaken that host guard.

In [ ]:
model_id = 'Qwen/Qwen3-8B'
revision = 'b968826d9c46dd6066d109eabc6255188de91218'
model_env = os.environ.copy()
for name in ('PYTHONHOME', 'PYTHONPATH'):
    model_env.pop(name, None)
Path('/content/iam_tmp').mkdir(exist_ok=True)
model_env.update(HF_HOME='/content/iam_huggingface', TMPDIR='/content/iam_tmp',
                 CUDA_VISIBLE_DEVICES='0', VLLM_USE_V1='0', VLLM_ATTENTION_BACKEND='XFORMERS',
                 VLLM_NO_USAGE_STATS='1', DO_NOT_TRACK='1', TOKENIZERS_PARALLELISM='false',
                 OMP_NUM_THREADS='2', MKL_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2')
download_code = "from huggingface_hub import snapshot_download; snapshot_download('Qwen/Qwen3-8B', revision='b968826d9c46dd6066d109eabc6255188de91218', allow_patterns=['*.json','*.safetensors','*.jinja','*.txt'], max_workers=2)"
subprocess.check_call([str(model_python), '-c', download_code], env=model_env)
snapshot = Path(model_env['HF_HOME']) / 'hub/models--Qwen--Qwen3-8B/snapshots' / revision
if not list(snapshot.glob('*.safetensors')):
    raise RuntimeError('The pinned model download is incomplete.')
if 'model_process' in globals() and model_process.poll() is None:
    raise RuntimeError('This notebook already has a model process. Do not launch a second one.')
with socket.socket() as port_check:
    port_check.bind(('127.0.0.1', 8000))
model_command = [str(model_python), '-m', 'vllm.entrypoints.openai.api_server',
    '--model', str(snapshot), '--served-model-name', model_id, '--dtype', 'half',
    '--host', '127.0.0.1', '--port', '8000', '--max-model-len', '4096',
    '--max-num-seqs', '1', '--tensor-parallel-size', '1',
    '--gpu-memory-utilization', '0.80', '--enforce-eager',
    '--disable-log-requests', '--disable-uvicorn-access-log', '--guided-decoding-backend', 'xgrammar']
model_log = Path('/content/iam_colab_vllm.log')
with model_log.open('w') as log:
    model_process = subprocess.Popen(model_command, env=model_env, stdin=subprocess.DEVNULL,
                                     stdout=log, stderr=log)
print('Model loading started. Run the next cell and wait for readiness.')


In [ ]:
local_http = urllib.request.build_opener(urllib.request.ProxyHandler({}))
deadline = time.monotonic() + 600
ready = False
last_notice = 0
while time.monotonic() < deadline:
    if model_process.poll() is not None:
        raise RuntimeError('The model process exited. Inspect /content/iam_colab_vllm.log; no readiness claimed.')
    if tunnel.poll() is not None:
        raise RuntimeError('The SSH tunnel stopped. Re-establish it before connecting the app.')
    try:
        with local_http.open('http://127.0.0.1:8000/health', timeout=3) as response:
            response.read()
        with local_http.open('http://127.0.0.1:8000/v1/models', timeout=3) as response:
            served = json.load(response)
        ready = any(row.get('id') == model_id for row in served.get('data', []))
    except (OSError, ValueError):
        ready = False
    if ready:
        break
    if time.monotonic() - last_notice >= 15:
        print('Waiting for model loading...', flush=True)
        last_notice = time.monotonic()
    time.sleep(2)
if not ready:
    raise RuntimeError('Readiness timed out. Inspect /content/iam_colab_vllm.log before retrying.')
Path('/content/iam_colab_run.json').write_text(json.dumps({
    'model': model_id, 'model_revision': revision, 'gpu': gpu_name,
    'command': model_command, 'validation': 'health_and_model_alias_only',
    'privacy_screening': 'existing_application_on_davinci',
    'remote_loopback_port': 18000, 'model_loopback_port': 8000
}, indent=2))
print('Local model health/model alias passed. Verify the forwarded endpoint and actual application on davinci next.')


## 5. Connect the existing UI on davinci

Run these commands in the **davinci terminal**, not in Colab. The conversational collection must already have been built locally from the supplied originals, as in the existing meeting guide. Keep the notebook connected while using the app.

```bash
cd ~/iam-sra-demo
IAM_BASE_URL=http://127.0.0.1:18000 scripts/health-check.sh
scripts/stop-ui.sh
IAM_MOCK=0 IAM_BASE_URL=http://127.0.0.1:18000 \
IAM_ASSESSMENT_MODE=retrieval_grounded_conversation scripts/start-ui.sh
```

Wait a few seconds, then check again:

```bash
IAM_BASE_URL=http://127.0.0.1:18000 scripts/health-check.sh
```

Both services must be healthy. Open the usual UI at `http://127.0.0.1:8501` through your existing access method. If using a laptop rather than davinci's own browser, retain your existing SSH/VS Code port forward; localhost on the laptop does not automatically refer to davinci. Start a fresh session so it uses the new endpoint.

Health/model-list checks alone do not establish tokenization, guided JSON or semantic accuracy. Exercise the three synthetic meeting cases through the real application:

```bash
IAM_BASE_URL=http://127.0.0.1:18000 PYTHONPATH=src \
.venv-privacy/bin/python scripts/smoke-baseline.py --conversation \
  --benchmark eval/conversation_meeting_cases.json \
  --output .runtime/conversation-colab-live.json
```

Inspect initial interpretations, question relevance, exports and errors. Scripted confirmation is automation, not citizen validation. Keep benchmark files on davinci. Never substitute successful HTTP/JSON checks for semantic review.

## 6. Finish the temporary session

This cell stops only the model and SSH subprocesses created by this notebook. It does not send signals to davinci processes. Remove the exact temporary public-key line you added to davinci's `~/.ssh/authorized_keys`, then disconnect/delete the Colab runtime. Session deletion is not a promise of forensic erasure. Colab disconnection stops inference; the UI must then point to a different working model endpoint before use.

In [ ]:
for name in ('model_process', 'tunnel'):
    process = globals().get(name)
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=10)
if 'key_path' in globals():
    key_path.unlink(missing_ok=True)
    Path(str(key_path) + '.pub').unlink(missing_ok=True)
print('Notebook-owned processes stopped and temporary key files removed. Remove its authorized-key line on davinci.')
